In [ ]:
##################################################### Attatched python code is excecuted in google colab ######################################################################
########################################################################################################################################################################################

#### Norimal inverse gamma distribution
from numpy import sum, mean, size, sqrt
from scipy.stats import norm, invgamma

def draw_mus_and_sigmas(data,m0,k0,s_sq0,v0,n_samples=10000):
    # number of samples
    N = size(data)
    # find the mean of the data
    the_mean = mean(data)
    # sum of squared differences between data and mean
    SSD = sum( (data - the_mean)**2 )

    # combining the prior with the data - page 79 of Gelman et al.
    # to make sense of this note that
    # inv-chi-sq(v,s^2) = inv-gamma(v/2,(v*s^2)/2)
    kN = float(k0 + N)
    mN = (k0/kN)*m0 + (N/kN)*the_mean
    vN = v0 + N
    vN_times_s_sqN = v0*s_sq0 + SSD + (N*k0*(m0-the_mean)**2)/kN
    s_sqN = vN_times_s_sqN/vN

    # 1) draw the variances from an inverse gamma
    # (params: alpha, beta)
    alpha = vN/2
    beta = vN_times_s_sqN/2
    # thanks to wikipedia, we know that:
    # if X ~ inv-gamma(a,1) then b*X ~ inv-gamma(a,b)
    sig_sq_samples = beta*invgamma.rvs(alpha,size=n_samples)

    # 2) draw means from a normal conditioned on the drawn sigmas
    # (params: mean_norm, var_norm)
    mean_norm = mN
    var_norm = sqrt(sig_sq_samples)/kN
    mu_samples = norm.rvs(mean_norm,scale=var_norm,size=n_samples)

    # 3) return the mu_samples and sig_sq_samples
    return kN,mN,vN,s_sqN,mu_samples, sig_sq_samples

In [ ]:
#### A sampling test from normal inverse gamma distribution
from numpy.random import normal
from matplotlib import pyplot as plt
from statistics import median

# step 1: define prior parameters for the normal and inverse gamma
m0 = 0.
k0 = 1.
s_sq0 = 1.
v0 = 1.

# step 2: get some random data, with slightly different statistics
data1 = normal(loc=5, scale=1, size=100)
#data2 = normal(loc=-1, scale=1, size=10)
##data3 = normal(loc=15, scale=1, size=100)
#data4 = normal(loc=4, scale=1, size=10)
#data5 = normal(loc=8, scale=1, size=10)
##data6 = normal(loc=7.5, scale=1, size=10)
#data7 = normal(loc=7.8, scale=1, size=10)



# step 3: get posterior samples
k1,m1,v1,s_sq1,mus1,sig_sqs1 = draw_mus_and_sigmas(data1,m0,k0,s_sq0,v0)
plt.plot(sig_sqs1)
plt.figure()
plt.hist(sig_sqs1,bins=20)
##mus2,sig_sqs2 = draw_mus_and_sigmas(data2,m0,k0,s_sq0,v0)
#mus3,sig_sqs3 = draw_mus_and_sigmas(data3,m0,k0,s_sq0,v0)
#mus4,sig_sqs4 = draw_mus_and_sigmas(data4,m0,k0,s_sq0,v0)
#mus5,sig_sqs5 = draw_mus_and_sigmas(data5,m0,k0,s_sq0,v0)
#mus6,sig_sqs6 = draw_mus_and_sigmas(data6,m0,k0,s_sq0,v0)
#mus7,sig_sqs7 = draw_mus_and_sigmas(data7,m0,k0,s_sq0,v0)


In [ ]:
#### Bayesian optimization

!pip install Gpy==1.9.8
!pip install GpyOpt==1.2.6
import numpy as np

import GPy
import GPyOpt

from GPyOpt.methods import BayesianOptimization


np.random.seed(1)
#Continous domain
#bounds = np.array([0,1000])

#Discrete domain
base = 2
max = 1000
x = np.arange(0,15)
bounds  =  max/(base**x)

##Matern kernel for GP
kernel = GPy.kern.Matern52(input_dim=5, variance=1.0, lengthscale=1.0)

##Bound for five lipid
bds = [{'name': 'Lipid', 'type': 'discrete', 'domain': bounds,'dimensionality': 5}]

#####Constraint: Manully add constraints sequentially to block the previous suggested combinations and generate a new one.  #####
con = [{'name':'constr_1','constraint':'-(x[:,1]-1000.0)'},
       {'name':'constr_2','constraint':'x[:,2]-125.0+0.000001'},
       {'name':'constr_3','constraint':'x[:,3]-125.0+0.000001'},

       ]


##### We manually add the observed X and y in the initial samples of GP for each run to sequentially optimized the combination. The first three are the initial samples randomly tried.#####
X = np.array([
[15,	120,	7.5,	7.5,	7.5],
[1.875,	60,	7.5,	7.5,	7.5],
[7.5,	7.5,	30,	30,	30],

[250.0,0.1220703125,500.0,250.0,7.8125],
[31.25,0.1220703125,125.0,62.5,15.625],
[0.244140625,0.244140625,7.8125,250.0,125.0],
[250.0,0.1220703125,125.0,0.1220703125,250.0],
[0.244140625,250.0,250.0,0.06103515625,62.5],
[7.8125,0.9765625,0.9765625,0.48828125,125.0],
[0.9765625,62.5,250.0,0.1220703125,500.0],
[31.25,1000.0,3.90625,7.8125,31.25],
[0.06103515625,0.9765625,0.48828125,31.25,1000.0],


[1.953125,0.244140625,500.0,1.953125,500.0],
[31.25,0.48828125,0.9765625,0.9765625,0.06103515625],
[15.625,3.90625,1000.0,500.0,3.90625],

[1.953125,125.0,0.244140625,0.244140625,31.25],
[1.953125,62.5,0.9765625,1000.0,7.8125],
[0.06103515625,1.953125,31.25,0.1220703125,62.5],

[0.1220703125,15.625,3.90625,125.0,0.06103515625],
[125.0,500.0,250.0,250.0,1000.0],
[7.8125,31.25,62.5,0.06103515625,125.0],


[250.0,250.0,1.953125,1000.0,62.5],
[250.0,0.244140625,0.1220703125,250.0,125.0],
[7.8125,1000.0,15.625,7.8125,125.0],

[1000,0.244140625,1000,0.06103515625,250],
[0.244140625,1000,0.9765625,0.48828125,62.5],
[125.0,125.0,3.90625,3.90625,250.0],

[0.48828125,0.9765625,7.8125,0.9765625,3.90625],
[62.5,15.625,0.9765625,0.9765625,0.06103515625],
[15.625,500.0,7.8125,1.953125,3.90625],

[7.8125,7.8125,0.48828125,125.0,7.8125],
[500.0,1000.0,0.06103515625,0.06103515625,0.48828125],
[125.0,0.06103515625,1.953125,0.9765625,1000.0],


[3.90625,31.25,0.9765625,250.0,3.90625],
[1000.0,0.06103515625,31.25,0.9765625,0.9765625],
[31.25,0.9765625,31.25,7.8125,0.1220703125],

[125.0, 0.06103515625, 500.0, 62.5, 62.5],
[15.625, 125.0, 500.0, 250.0, 125.0],
[0.9765625, 250.0, 250.0, 0.1220703125, 15.625],

[15.625,3.90625,7.8125,15.625,500.0],
[15.625,31.25,0.06103515625,0.48828125, 62.5],
[0.06103515625,1000.0,15.625,125.0,3.90625],

[0.244140625, 15.625, 500.0, 1000.0, 0.06103515625],
[0.06103515625, 0.06103515625, 0.244140625, 500.0, 15.625],
[62.5, 0.1220703125, 1000.0, 3.90625, 31.25],


[1.953125, 0.9765625, 0.48828125, 0.244140625, 1000.0],
[3.90625, 125.0, 125.0, 500.0, 500.0],
[500.0, 125.0, 0.244140625, 0.1220703125, 7.8125],

[1000.0,1000.0,7.8125,31.25,15.625],
[62.5,1000.0,0.48828125,1.953125,31.25],
[62.5,1000.0,0.1220703125,3.90625,7.8125],

[500.0,1000.0,31.25,1.953125,1000.0],
[0.48828125,1000.0,0.244140625,0.244140625,31.25],
[31.25,1000.0,7.8125,0.9765625,250.0],

[125.0, 1000.0, 0.9765625, 0.244140625, 15.625],
[0.244140625, 1000.0, 31.25, 15.625, 7.8125],
[125.0,1000.0,1.953125,62.5,500.0],

[0.06103515625,1000.0,0.244140625,0.48828125,0.06103515625],
[250.0,1000.0,3.90625,0.1220703125,0.48828125],
[0.1220703125,1000.0,15.625,0.244140625,250.0],


[0.48828125, 1000.0, 62.5, 7.8125, 0.1220703125],
[1000.0, 1000.0, 1.953125, 0.244140625, 125.0],
[125.0, 1000.0, 62.5, 0.1220703125, 500.0],

[250.0,1000.0,1.953125,0.244140625,125.0],

[1000.0,1000.0,0.9765625,0.06103515625,0.48828125],

[1000.0,1000.0,31.25,0.06103515625,500.0],



[250.0,1000.0,0.1220703125,0.9765625,31.25],


[31.25,1000.0,0.48828125,31.25,0.48828125],

[250.0,1000.0,31.25,0.244140625,0.1220703125],


[0.9765625,1000.0,31.25,62.5,3.90625],
[1000.0,1000.0,1.953125,62.5,500.0],
[62.5,1000.0,31.25,15.625,250.0],


[3.90625,1000.0,62.5,0.06103515625,62.5],
[125.0,1000.0,15.625,15.625,125.0],
[3.90625,1000.0,1.953125,3.90625,1000.0],

[3.90625,1000.0,62.5,1.953125,1000.0],
[0.9765625,1000.0,3.90625,31.25,7.8125],
[62.5,1000.0,0.1220703125,62.5,7.8125],

[7.8125,1000.0,0.9765625,62.5,31.25],
[31.25,1000.0,0.06103515625,0.06103515625,0.9765625],
[7.8125,1000.0,62.5,31.25,7.8125],


[0.48828125,1000.0,62.5,62.5,0.48828125],
[31.25,1000.0,0.9765625,62.5,0.48828125],
[31.25,1000.0,62.5,0.244140625,15.625],


[ 62.5,1000.0,3.90625,31.25,500.0],
[ 125.0,1000.0,1.953125,15.625,500.0],
[3.90625,1000.0,7.8125,31.25,7.8125],

[0.06103515625,1000.0,0.06103515625,0.06103515625,500.0]



])

y = np.array([
1.048824131,
1.019156664,
1.003246445,

1.859133429,
1.728314403,
1.486032029,

1.340374067,
1.291074878,
1.241435245,

0.737352393,
1.072208218,
1.475318643,

1.390809871,
1.129947411,
1.802342897,

1.165039202,
1.070355892,
1.067391564,

1.175026627,
2.001202681,
1.273685468,


1.403225241,
1.228475345,
0.970363522,

1.356263681,
0.942686627,
0.96697517,

1.187786093,
1.332546874,
1.146295319,


1.042671082,
0.896783996,
1.154858516,

1.127543065,
1.42114431,
1.255466996,

1.431245026,
1.780268116,
1.426882571,


1.072903122,
1.303509767,
0.718979632,

1.353187628,
1.331260153,
1.629187232,

1.003715271,
1.829033333,
1.542589328,


0.941977352,
0.614944568,
0.946972622,

1.091737437,
0.774839267,
0.596975675,

0.870516692,
0.706652393,
0.719849778,

0.629092735,
0.711292765,
0.657801492,

0.738030993,
0.764355007,
1.034278979,

0.981541443,
1.244570874,
1.071772202,

0.615097421,
0.751305165,
0.528081684,

0.734833139,
0.770716911,
0.714442282,

0.868842763,
1.060972185,
0.874040168,

0.968961489,
1.086964281,
0.749257998,

0.791713156,
0.774780143,
0.771451801,

0.527232383,
0.465775984,
0.429423618,

0.43819734,
0.327400671,
0.635108258,

0.437253644,


]
)


##### Manully input the newly-generated samples (9 samples per run: 3 combinations and each with 3 replicates) #####
y_update= np.array([
 1.157341307,
 0.936019749,
 1.053111336,
 0.96783278,
 1,
 1.089637212,
 0.925933613,
 1.101246199,
 0.982559522,


]
)

##### Manully change the 4 parameters on the updated (posterior) normal inverse gammma distribution for each run except for initial run (the reuslts will be shown in the console)  #####
#
#Previous posterior as prior
#m0 = 1.0486222511221375
#k0 = 262.0
#s_sq0 = 0.15488160222723504
#v0 = 262.0

k,m,v,s_sqn,mus,sig_sqs = draw_mus_and_sigmas(y_update,m0,k0,s_sq0,v0)
print('medain of mu and sig_sqs: ',median(mus),median(sig_sqs))
print('posterior pararmeter: ',k,m,v,s_sqn)

noise=np.sqrt(median(sig_sqs))

def simple_f(X,noise=noise):
  score = 0+ np.random.normal(0, noise, 1)

  return score

optimizer = GPyOpt.methods.BayesianOptimization(f=simple_f,
                                 domain=bds,
                                 constraints = con,
                                 model_type='GP',
                                 kernel=kernel,
                                 acquisition_type ='EI',
                                 ##### Manually adjust the jitter for each run execpt for the initial run (The initial jitter we set is 0.01)#####
                                 acquisition_jitter = 0.0001348,

                                 X=X,
                                 Y=y.reshape(-1,1),
                                 noise_var= median(sig_sqs),

                                 exact_feval=False,
                                 normalize_Y=False,
                                 maximize=False)

optimizer.run_optimization(max_iter=0)

print('The combination in the history:', optimizer.X )
print('The scores for all the combinations in the history:', optimizer.Y )
print('The best combination in the history:', optimizer.X[optimizer.Y.argmin()] )
print('The best score for all the combinations in the history:',min(optimizer.Y)  )

next_combination = optimizer.suggest_next_locations()
print('The next combination would be:',next_combination)

print('A: ',next_combination[0][0] )
print('B: ',next_combination[0][1] )
print('C: ',next_combination[0][2] )
print('D: ',next_combination[0][3] )
print('E: ',next_combination[0][4] )



In [ ]:
##### The code for dynamic aquisition jitter for next run
import itertools

bounds = list(bounds)
a = [bounds,bounds,bounds,bounds,bounds]
combination_a = list(itertools.product(*a))

combination_a = np.array(combination_a)

sum_var = 0
time = 0

for i in combination_a:
  sum_var = (sum_var + (optimizer.model.predict(i)[1])**2)
  time += 1

average_var = sum_var/time

f_star = min(optimizer.Y)

jitter =(average_var/(1/f_star))/(1.22652742*100)
print(jitter)
#print( optimizer.model.predict( np.array([[15.625, 62.5, 250.0, 7.8125, 0.244140625]]) )[1] )
#np.shape(next_combination)
#print(next_combination)

In [ ]:
#### The plot for the results
####The python code for plots.

from scipy.spatial import distance
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import matplotlib.pyplot as plt

X = np.array([
[15,	120,	7.5,	7.5,	7.5],
[1.875,	60,	7.5,	7.5,	7.5],
[7.5,	7.5,	30,	30,	30],

[250.0,0.1220703125,500.0,250.0,7.8125],
[31.25,0.1220703125,125.0,62.5,15.625],
[0.244140625,0.244140625,7.8125,250.0,125.0],
[250.0,0.1220703125,125.0,0.1220703125,250.0],
[0.244140625,250.0,250.0,0.06103515625,62.5],
[7.8125,0.9765625,0.9765625,0.48828125,125.0],
[0.9765625,62.5,250.0,0.1220703125,500.0],
[31.25,1000.0,3.90625,7.8125,31.25],
[0.06103515625,0.9765625,0.48828125,31.25,1000.0],


[1.953125,0.244140625,500.0,1.953125,500.0],
[31.25,0.48828125,0.9765625,0.9765625,0.06103515625],
[15.625,3.90625,1000.0,500.0,3.90625],

[1.953125,125.0,0.244140625,0.244140625,31.25],
[1.953125,62.5,0.9765625,1000.0,7.8125],
[0.06103515625,1.953125,31.25,0.1220703125,62.5],

[0.1220703125,15.625,3.90625,125.0,0.06103515625],
[125.0,500.0,250.0,250.0,1000.0],
[7.8125,31.25,62.5,0.06103515625,125.0],


[250.0,250.0,1.953125,1000.0,62.5],
[250.0,0.244140625,0.1220703125,250.0,125.0],
[7.8125,1000.0,15.625,7.8125,125.0],

[1000,0.244140625,1000,0.06103515625,250],
[0.244140625,1000,0.9765625,0.48828125,62.5],
[125.0,125.0,3.90625,3.90625,250.0],

[0.48828125,0.9765625,7.8125,0.9765625,3.90625],
[62.5,15.625,0.9765625,0.9765625,0.06103515625],
[15.625,500.0,7.8125,1.953125,3.90625],

[7.8125,7.8125,0.48828125,125.0,7.8125],
[500.0,1000.0,0.06103515625,0.06103515625,0.48828125],
[125.0,0.06103515625,1.953125,0.9765625,1000.0],


[3.90625,31.25,0.9765625,250.0,3.90625],
[1000.0,0.06103515625,31.25,0.9765625,0.9765625],
[31.25,0.9765625,31.25,7.8125,0.1220703125],

[125.0, 0.06103515625, 500.0, 62.5, 62.5],
[15.625, 125.0, 500.0, 250.0, 125.0],
[0.9765625, 250.0, 250.0, 0.1220703125, 15.625],

[15.625,3.90625,7.8125,15.625,500.0],
[15.625,31.25,0.06103515625,0.48828125, 62.5],
[0.06103515625,1000.0,15.625,125.0,3.90625],

[0.244140625, 15.625, 500.0, 1000.0, 0.06103515625],
[0.06103515625, 0.06103515625, 0.244140625, 500.0, 15.625],
[62.5, 0.1220703125, 1000.0, 3.90625, 31.25],


[1.953125, 0.9765625, 0.48828125, 0.244140625, 1000.0],
[3.90625, 125.0, 125.0, 500.0, 500.0],
[500.0, 125.0, 0.244140625, 0.1220703125, 7.8125],

[1000.0,1000.0,7.8125,31.25,15.625],
[62.5,1000.0,0.48828125,1.953125,31.25],
[62.5,1000.0,0.1220703125,3.90625,7.8125],

[500.0,1000.0,31.25,1.953125,1000.0],
[0.48828125,1000.0,0.244140625,0.244140625,31.25],
[31.25,1000.0,7.8125,0.9765625,250.0],

[125.0, 1000.0, 0.9765625, 0.244140625, 15.625],
[0.244140625, 1000.0, 31.25, 15.625, 7.8125],
[125.0,1000.0,1.953125,62.5,500.0],

[0.06103515625,1000.0,0.244140625,0.48828125,0.06103515625],
[250.0,1000.0,3.90625,0.1220703125,0.48828125],
[0.1220703125,1000.0,15.625,0.244140625,250.0],


[0.48828125, 1000.0, 62.5, 7.8125, 0.1220703125],
[1000.0, 1000.0, 1.953125, 0.244140625, 125.0],
[125.0, 1000.0, 62.5, 0.1220703125, 500.0],

[250.0,1000.0,1.953125,0.244140625,125.0],

[1000.0,1000.0,0.9765625,0.06103515625,0.48828125],

[1000.0,1000.0,31.25,0.06103515625,500.0],



[250.0,1000.0,0.1220703125,0.9765625,31.25],


[31.25,1000.0,0.48828125,31.25,0.48828125],

[250.0,1000.0,31.25,0.244140625,0.1220703125],


[0.9765625,1000.0,31.25,62.5,3.90625],
[1000.0,1000.0,1.953125,62.5,500.0],
[62.5,1000.0,31.25,15.625,250.0],


[3.90625,1000.0,62.5,0.06103515625,62.5],
[125.0,1000.0,15.625,15.625,125.0],
[3.90625,1000.0,1.953125,3.90625,1000.0],

[3.90625,1000.0,62.5,1.953125,1000.0],
[0.9765625,1000.0,3.90625,31.25,7.8125],
[62.5,1000.0,0.1220703125,62.5,7.8125],

[7.8125,1000.0,0.9765625,62.5,31.25],
[31.25,1000.0,0.06103515625,0.06103515625,0.9765625],
[7.8125,1000.0,62.5,31.25,7.8125],


[0.48828125,1000.0,62.5,62.5,0.48828125],
[31.25,1000.0,0.9765625,62.5,0.48828125],
[31.25,1000.0,62.5,0.244140625,15.625],


[ 62.5,1000.0,3.90625,31.25,500.0],
[ 125.0,1000.0,1.953125,15.625,500.0],
[3.90625,1000.0,7.8125,31.25,7.8125],

[0.06103515625,1000.0,0.06103515625,0.06103515625,500.0]





])

y = np.array([
1.048824131,
1.019156664,
1.003246445,

1.859133429,
1.728314403,
1.486032029,

1.340374067,
1.291074878,
1.241435245,

0.737352393,
1.072208218,
1.475318643,

1.390809871,
1.129947411,
1.802342897,

1.165039202,
1.070355892,
1.067391564,

1.175026627,
2.001202681,
1.273685468,


1.403225241,
1.228475345,
0.970363522,

1.356263681,
0.942686627,
0.96697517,

1.187786093,
1.332546874,
1.146295319,


1.042671082,
0.896783996,
1.154858516,

1.127543065,
1.42114431,
1.255466996,

1.431245026,
1.780268116,
1.426882571,


1.072903122,
1.303509767,
0.718979632,

1.353187628,
1.331260153,
1.629187232,

1.003715271,
1.829033333,
1.542589328,


0.941977352,
0.614944568,
0.946972622,

1.091737437,
0.774839267,
0.596975675,

0.870516692,
0.706652393,
0.719849778,

0.629092735,
0.711292765,
0.657801492,

0.738030993,
0.764355007,
1.034278979,

0.981541443,
1.244570874,
1.071772202,

0.615097421,
0.751305165,
0.528081684,

0.734833139,
0.770716911,
0.714442282,

0.868842763,
1.060972185,
0.874040168,

0.968961489,
1.086964281,
0.749257998,

0.791713156,
0.774780143,
0.771451801,

0.527232383,
0.465775984,
0.429423618,

0.43819734,
0.327400671,
0.635108258,

0.437253644,


]
)



dist = []

for i in range(0,np.shape(X)[0]-1):
  dist.append(distance.euclidean(X[i],X[i+1]))

print(dist)
print("____________")
distt = []
c= 1
for i in dist:

  if ((c%3!=0)):
    distt.append(i)
  c += 1

print(distt)

distt = np.array(distt)

dist_best = distt.reshape(29,2)

dist_best = np.mean(dist_best,axis=1)

plt.figure(figsize=(10, 10))
plt.plot (np.arange(0,16),dist_best[0:16],'b-')
plt.plot (np.arange(15,29),dist_best[15:29],'r-')

plt.scatter(np.arange(0,16),dist_best[0:16], color = 'blue')
plt.scatter(np.arange(16,29),dist_best[16:29], color = 'red')
plt.grid()
plt.xlabel('days',fontsize = 20)
plt.ylabel('distance(combination[n],combination[n-1])',fontsize = 20)
plt.title("Distance between consecutive combinations",fontsize = 20)
plt.legend(["BO","BO with adjustment"], loc ="upper right",fontsize=15)

distt = np.array(dist)

dist_best = distt.reshape(87,1)

dist_best = np.mean(dist_best,axis=1)

plt.figure(figsize=(10, 10))
plt.plot (np.arange(0,48),dist_best[0:48],'b-')
plt.plot (np.arange(47,87),dist_best[47:87],'r-')

plt.scatter(np.arange(0,48),dist_best[0:48], color = 'blue')
plt.scatter(np.arange(47,87),dist_best[47:87], color = 'red')

plt.grid()
plt.xlabel('iterations',fontsize = 20)
plt.ylabel('distance(combination[n],combination[n-1])',fontsize = 20)
plt.title("Distance between consecutive combinations",fontsize = 20)
plt.legend(["BO","BO with adjustment"], loc ="upper right",fontsize=15)
y_value = []

best_y = 10

for i in range(len(y)):
  if y[i]<best_y:
    best_y = y[i]
  y_value.append(best_y)

directed_evolution = [0.869610228]*len(y_value)

color_1=dist_best[0:48]
color_2=dist_best[47:87]
#color_3=dist_best

color_3=np.hstack((np.array([0]),dist_best))

plt.figure(figsize=(10, 10))
#plt.plot (np.arange(0,48),y_value[0:48],'b-')
#plt.plot (np.arange(48,len(y_value)),y_value[48:len(y_value)],'r-')
plt.plot (np.arange(0,48),directed_evolution[0:48],'r--',linewidth=4)
plt.scatter(np.arange(0,len(y_value)),directed_evolution, color = 'red')
plt.scatter(np.arange(0,48),y_value[0:48], s=200, c=color_1,cmap='viridis')
cbar=plt.colorbar(orientation='vertical')
cbar.set_label(label='Consecutive distance',size=20)
plt.legend(["Best value discovered in directed evolution"], bbox_to_anchor=(1, 0.6),loc ="center right",fontsize=15)
plt.grid()
plt.xlabel('iterations',fontsize = 20)
plt.ylabel('normalized absorbance (%)',fontsize = 20)
plt.title("The best discoved normalized absorbance (BO)",fontsize = 20)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)



plt.figure(figsize=(10, 10))
#plt.plot (np.arange(48,len(y_value)),directed_evolution[48:len(y_value)],'r--',linewidth=4)
plt.scatter(np.arange(48,len(y_value)),y_value[48:len(y_value)], s=200, c=color_2,cmap='plasma')
cbar=plt.colorbar(orientation='vertical')
cbar.set_label(label='Consecutive distance',size=20)

#plt.legend(["Best value discovered in directed evolution"], loc ="upper right",fontsize=15)
plt.grid()
plt.xlabel('iterations',fontsize = 20)
plt.ylabel('normalized absorbance (%)',fontsize = 20)
plt.title("The best discoved normalized absorbance (Adjusted BO)",fontsize = 20)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)


plt.figure(figsize=(20, 10))
plt.plot (np.arange(0,len(y_value)),directed_evolution[0:len(y_value)],'r--',linewidth=4)
orig_map=plt.cm.get_cmap('coolwarm')
reversed_map = orig_map.reversed()
plt.scatter(np.arange(0,len(y_value)),y_value[0:len(y_value)], s=200,color="blue")
plt.plot(y_value[0:len(y_value)],'b-')

n = 1.5
gradient = np.linspace(0, 1, 1000).reshape(1, -1)
plt.imshow(gradient , extent=[-1, len(y_value), 0.3, 1.1], aspect='auto', cmap='coolwarm',alpha=0.6)
#plt.scatter(np.arange(0,len(y_value)),y_value[0:len(y_value)], s=200, c=color_3,cmap=reversed_map)
cbar=plt.colorbar(orientation='horizontal',alpha=0.6)
cbar.set_label(label='Exploration \u2190                                            \u2192 Exploitation',size=20)



#plt.legend(["Best value discovered in directed evolution"], loc ="upper right",fontsize=15)

plt.grid()
plt.xlabel('iterations',fontsize = 20)
plt.ylabel('normalized absorbance (%)',fontsize = 20)
plt.title("The best discoved normalized absorbance",fontsize = 20)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)
plt.legend(["Best value discovered in directed evolution"], loc ="upper right",fontsize=15)


